# Módulo 7 — Detección de anomalías y cambios en el proceso

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Distinguir pico aislado de cambio persistente; métodos robustos, límites dinámicos, residuos del modelo, CUSUM y change-point.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # gráficos estadísticos (preinstalado en Colab)

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

Nota: este notebook usa la serie **verdadera**. Para ver anomalías de sensor reales, repite los pasos sobre `datos_proceso_planta.csv` (versión de campo).

In [ ]:
serie = df['Recuperacion_pct'].dropna()

## 2. Z-score global vs. robusto (mediana / MAD)

In [ ]:
z = (serie - serie.mean()) / serie.std()
med = serie.median()
mad = (serie - med).abs().median()
z_rob = 0.6745 * (serie - med) / mad
print('|z|>3      :', int((z.abs() > 3).sum()))
print('|z_rob|>3.5:', int((z_rob.abs() > 3.5).sum()))

## 3. Límites dinámicos con estadística móvil

In [ ]:
w = 168
mu = serie.rolling(w, center=True).mean()
sg = serie.rolling(w, center=True).std()
sup, inf = mu + 3*sg, mu - 3*sg
fuera = serie[(serie > sup) | (serie < inf)]

fig, ax = plt.subplots(figsize=(12,4))
serie.plot(ax=ax, alpha=0.5)
mu.plot(ax=ax, color='k'); sup.plot(ax=ax, ls='--', color='r'); inf.plot(ax=ax, ls='--', color='r')
ax.scatter(fuera.index, fuera.values, color='red', zorder=5)
plt.title(f'{len(fuera)} puntos fuera de banda dinámica'); plt.show()

## 4. Filtro de Hampel

In [ ]:
def hampel(x, w=12, n_sig=3.0):
    x = x.copy()
    med = x.rolling(2*w+1, center=True).median()
    mad = (x - med).abs().rolling(2*w+1, center=True).median()
    umbral = n_sig * 1.4826 * mad
    return (x - med).abs() > umbral

marca = hampel(serie)
print('anomalías Hampel:', int(marca.sum()))

## 5. Residuos de un modelo como detector

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
n_train = int(len(serie)*0.7)
m = SARIMAX(serie.iloc[:n_train], order=(2,1,1), seasonal_order=(1,0,1,24),
            enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred = m.get_forecast(steps=len(serie)-n_train)
obs = serie.iloc[n_train:]
e = obs - pred.predicted_mean
ci = pred.conf_int()
fuera_pi = obs[(obs < ci.iloc[:,0]) | (obs > ci.iloc[:,1])]

fig, ax = plt.subplots(figsize=(12,4))
obs.plot(ax=ax, label='observado')
pred.predicted_mean.plot(ax=ax, label='esperado')
ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.2)
ax.scatter(fuera_pi.index, fuera_pi.values, color='red', zorder=5)
ax.legend(); plt.title('Observaciones fuera del intervalo esperado'); plt.show()

## 6. Cambio persistente: CUSUM sobre los residuos

In [ ]:
en = (e - e.mean()) / e.std()
k = 0.5
sp = np.zeros(len(en)); sm = np.zeros(len(en))
for i in range(1, len(en)):
    sp[i] = max(0, sp[i-1] + en.iloc[i] - k)
    sm[i] = min(0, sm[i-1] + en.iloc[i] + k)
cusum = pd.DataFrame({'S+': sp, 'S-': sm}, index=en.index)
cusum.plot(title='CUSUM de residuos estandarizados'); plt.axhline(5, ls='--', color='r'); plt.show()

## 7. Change-point (cambio de régimen A→B)

In [ ]:
%pip -q install ruptures
import ruptures as rpt
y = df['Ley_Cu_pct'].dropna().values
algo = rpt.Pelt(model='rbf', min_size=200).fit(y)
bkps = algo.predict(pen=15)
idx = df['Ley_Cu_pct'].dropna().index
print('puntos de cambio detectados:')
for b in bkps[:-1]:
    print('  ', idx[b])
df['Ley_Cu_pct'].dropna().plot()
for b in bkps[:-1]:
    plt.axvline(idx[b], color='r', ls='--')
plt.title('Ley de Cu con puntos de cambio'); plt.show()

## Actividades sugeridas

1. Ejecuta el Z-score y el Hampel sobre `datos_proceso_planta.csv`: ¿detectan los picos de sensor y el tramo congelado?
2. ¿Qué método marca el cambio de régimen del día 112 y cuál solo marca los picos?
3. Ajusta el umbral de la banda dinámica (k=2,3,4) y discute falsas alarmas vs eventos perdidos.
4. Interpreta operacionalmente: ¿qué registros pedirías para explicar el change-point detectado?

---
## Cierre

El mismo modelo de pronóstico sirve para vigilar: observado − esperado → desviación → alerta → interpretación. El Módulo 8 integra todo en un caso.